# SimBa tutorial

Companion to *How Similar Are Two Brains? A Comprehensive Benchmark of Brain
Network Similarity Measures* (Dendorfer, Luppi, Poli, Mousley, Astle and Fakhar).

- **Part A** compares two networks with the 16 measures of the paper.
- **Part B** scores your own comparison function on the benchmark and ranks it
  against all 16.

```bash
pip install git+https://github.com/ad045/simba
```

In [1]:
import numpy as np
import pandas as pd

import simba_networks as sb

## Part A: comparing two networks

A network is a binary, symmetric (N, N) adjacency matrix without self-loops.
You can use your own, but if you do not have any at hand, this package ships the 25,000 networks of the paper's generative network model (GNM) sweep. They were generated as a 150 x 50 grid over the wiring parameters $\eta$ (distance
penalty) and $\gamma$ (matching-index reward), with 10 replicates per cell, and on 100
nodes (Schaefer-100) and with 495 edges each.

In [2]:
networks, meta = sb.load_networks()
networks.shape

(25000, 100, 100)

Take three of them: two networks `A1` and `A2` generated from the same
$(\eta, \eta)$ parameter combination, and a network `B` from the opposite corner of the parameter space.

In [3]:
cell = (meta.eta == meta.eta.unique()[22]) & (meta.gamma == meta.gamma.unique()[10])
A1, A2 = networks[cell].astype(float)[:2]
B = networks[(meta.eta == meta.eta.max()) & (meta.gamma == meta.gamma.min())][0].astype(float)

Every measure is a plain function `f(A, B) -> float`. In this library, `sb.MEASURES` holds all
16, keyed by name. `sb.NAMES` gives the name used in the paper.

In [4]:
sb.MEASURES["delta_con"](A1, B)

9.300131308649199

In [5]:
table = pd.DataFrame({
    "measure": [sb.NAMES[m] for m in sb.MEASURES],
    "similarity": [m in sb.SIMILARITIES for m in sb.MEASURES],
    "A1 vs A2": [f(A1, A2) for f in sb.MEASURES.values()],
    "A1 vs B": [f(A1, B) for f in sb.MEASURES.values()],
}, index=list(sb.MEASURES))
table.round(3)

/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:394: RuntimeWarning: divide by zero encountered in matmul
  eAw = eAw @ eAw
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:394: RuntimeWarning: overflow encountered in matmul
  eAw = eAw @ eAw
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:394: RuntimeWarning: invalid value encountered in matmul
  eAw = eAw @ eAw
/Users/adrian/Documents/01_projects/14_4D_lab/simba/simba_networks/measures.py:360: RuntimeWarning: divide by zero encountered in matmul
  tri = np.diagonal(M @ M @ M)
/Users/adrian/Documents/01_projects/14_4D_lab/simba/simba_networks/measures.py:360: RuntimeWarning: overflow encountered in matmul
  tri = np.diagonal(M @ M @ M)
/Users/adrian/Documents/01_projects/14_4D_lab/simba/simba_networks/measures.py:360: RuntimeWarning: invalid value encountered in matmul
  tri = np.diagonal(M @ M @ M)


,measure,similarity,A1 vs A2,A1 vs B
frobenius,Frobenius,False,30.332,39.699
hamming,Hamming,False,460.000,788.000
jaccard,Jaccard,True,0.366,0.114
f1,F1,True,0.535,0.204
network_mutual_information,NMI,True,0.225,0.017
dc_network_mutual_information,DC-NMI,True,0.247,0.030
spectral_distance_adjacency,Spectral (adjacency),False,4.472,13.380
spectral_distance_norm_laplacian,Spectral (normalized Laplacian),False,0.088,0.880
netrd_non_backtracking_spectral,Spectral (non-backtracking),False,0.210,0.971
portrait,Portrait divergence,False,0.180,0.577


Five measures are similarities (higher = more alike), the other eleven are
distances (lower = more alike). Negate the similarities before ranking or
averaging. Then every measure should place the replicate `A2` closer to `A`
than the far-off `B`:

In [6]:
sign = np.where(table["similarity"], -1, 1)
(sign * table["A1 vs A2"] < sign * table["A1 vs B"]).all()

np.True_

Your own networks go in the same way. The values depend on network size and
density, so only compare networks with the same nodes, and compare measures by
their rankings, never by their raw values.

## Part B: scoring your own measure

Write your measure as `f(A, B) -> float`, with `A` a generated network and `B`
the reference. Lower must mean more alike, or pass `similarity=True` to
`evaluate`. Return `nan` (or raise) when a comparison fails.

The example compares the two degree distributions with a Kolmogorov-Smirnov
statistic. It ignores where the edges sit, so expect it to do poorly where
topology matters.

In [7]:
from scipy.stats import ks_2samp

def degree_ks(A, B):
    """KS statistic between the degree distributions. Lower = more alike."""
    return float(ks_2samp(A.sum(0), B.sum(0)).statistic)

degree_ks(A1, A2), degree_ks(A1, B)

(0.07, 0.13)

### Accuracy

Accuracy is estibated by parameter recovery. Parameter recovery compares generated networks with generated networks, so it
needs no empirical data: 100 test networks at known parameters are matched to
100 grid points, once over the whole parameter space and once inside the
"plausible" window (a narrower range of $\eta$ and $\gamma$ values). The error is in grid steps (chance: 4.96 and 4.53).

`report.table` has one row per read-out, your value next to all 16 published
measures, and your `rank` among the 17 (1 = best). The two AUC rows (real
subjects against GNMs) stay empty for your measure unless you also drop in the
individual HCP connectomes.

In [8]:
report = sb.evaluate(degree_ks, name="degree KS", criteria=["accuracy"])
report.table.drop(columns="criterion").round(2)

/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parameter recovery, window: 100%|██████████| 40/40 [00:02<00:00, 18.04it/s]


,readout,direction,degree KS,Frobenius,Hamming,Jaccard,F1,NMI,DC-NMI,Spectral (adjacency),Spectral (normalized Laplacian),Spectral (non-backtracking),Portrait divergence,NetSimile,DeltaCon,Communicability correlation,Communicability JSD,Resistance distance,KS-based energy,rank
recovery_wide,"Recovery error, whole morphospace, grid steps",lower,4.78,3.71,3.71,3.18,3.18,3.31,4.83,3.67,4.21,4.09,3.77,3.44,2.74,3.26,3.25,3.54,3.23,16.0
recovery_window,"Recovery error, plausible window, grid steps",lower,6.00,4.01,4.01,4.09,4.09,4.09,4.02,5.26,6.28,6.68,3.37,3.72,2.36,3.63,3.85,4.57,6.29,14.0
r_eta_wide,"r(true eta, recovered eta), whole morphospace",higher,0.52,0.70,0.70,0.61,0.61,0.61,-0.17,0.57,0.36,0.49,0.56,0.72,0.82,0.45,0.75,0.77,0.76,13.0
r_eta_window,"r(true eta, recovered eta), plausible window",higher,-0.03,0.26,0.26,0.24,0.24,0.20,0.10,-0.01,-0.21,-0.28,0.06,0.08,0.13,0.19,0.22,-0.08,-0.01,14.0
auc,AUC real vs the full GNM population,higher,NaN,1.00,NaN,NaN,NaN,NaN,NaN,0.97,NaN,0.99,0.97,0.92,1.00,1.00,NaN,NaN,0.98,NaN
auc_hard,AUC real vs the own 500 best networks,higher,NaN,1.00,NaN,NaN,NaN,NaN,NaN,0.46,NaN,0.59,0.26,0.10,1.00,1.00,NaN,NaN,0.27,NaN


### The other four criteria

The other four criteria need an empirical reference connectome. 
Agreement, plausibility, efficiency, and sensitivity and robustness compare - in our case -
the 25,000 generated networks with the empirical HCP consensus connectome.

As we may not redistribute the HCP consensus connectome, you are required to save your copy as
`~/simba_networks_data/reference/consensus.npy` (binary, symmetric
(100, 100), Schaefer-100 order). For testing, if you do not have an empirical consensus connectome at hand, the next cell falls back to a stand-in (a held-out generated
network). 

The results then describe the
stand-in and not a brain, and are thus not comparable to the published columns.

In [9]:
try:
    reference = sb.load_reference()
except FileNotFoundError:
    print("No consensus found: using a generated stand-in. Numbers are NOT comparable to the paper.")
    reference = sb.load_networks("recovery_window")["targets"][0].astype(float)

report = sb.evaluate(degree_ks, name="degree KS", reference=reference,
                     criteria=["agreement", "plausibility", "efficiency"])
report.table.drop(columns="criterion").round(2)

/Users/adrian/Documents/01_projects/14_4D_lab/simba/simba_networks/benchmark.py:414: UserWarning: This reference is not the consensus the paper used: scores will not be comparable to the published ones.
  ref = data.load_reference(reference) if needs_reference else None


No consensus found: using a generated stand-in. Numbers are NOT comparable to the paper.


landscape (25,000 networks): 100%|██████████| 100/100 [00:05<00:00, 18.42it/s]


,readout,direction,degree KS,Frobenius,Hamming,Jaccard,F1,NMI,DC-NMI,Spectral (adjacency),Spectral (normalized Laplacian),Spectral (non-backtracking),Portrait divergence,NetSimile,DeltaCon,Communicability correlation,Communicability JSD,Resistance distance,KS-based energy,rank
agreement,Mean r with the selected landscapes,higher,0.23,0.46,0.53,0.53,0.53,0.52,0.51,0.57,0.65,0.61,0.37,0.55,0.43,0.27,-0.10,0.16,0.45,15.0
max_abs_r,Largest |r| with a published landscape,,0.83,1.00,1.00,1.00,1.00,0.99,0.98,0.85,0.93,0.93,0.65,0.77,0.64,0.53,0.71,0.71,0.84,NaN
best_eta,Best-fitting eta,,-4.41,-3.73,-3.73,-3.73,-3.73,-3.73,-3.29,-3.96,-3.06,-3.96,-3.06,-2.84,-3.73,-4.18,-0.59,-7.10,-1.27,NaN
best_gamma,Best-fitting gamma,,0.28,0.06,0.06,0.06,0.06,0.06,0.73,0.33,0.10,0.08,0.12,0.12,0.06,0.96,-0.08,0.64,0.21,NaN
eta_positive_of_100,"Combinations at eta > 0, of the 100 best",lower,20.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,4.00,0.00,0.00,18.00,27.00,18.00,16.0
eta_positive_of_20,"Combinations at eta > 0, of the 20 best",lower,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,7.00,0.00,8.0
total_variation,Total variation of the 20 best combinations,lower,0.08,0.08,0.08,0.08,0.08,0.08,0.03,0.02,0.00,0.05,0.01,0.01,0.00,0.11,0.13,0.24,0.00,14.0
connection_length_mm,Mean connection length of the 20 best networks...,,53.29,35.42,35.42,35.42,35.42,35.42,34.94,40.94,39.54,41.28,38.72,41.78,39.75,60.80,66.86,69.46,47.63,NaN
percent_beyond_90mm,"Connections beyond 90 mm, %",,19.49,1.98,1.98,1.98,1.98,1.98,1.36,6.16,5.01,7.22,3.37,7.41,4.59,25.45,30.46,33.37,10.53,NaN
degree_sd,Degree SD of the 20 best networks,,3.40,3.90,3.90,3.90,3.90,3.90,4.21,3.81,3.84,3.65,3.87,4.02,3.54,3.79,2.55,2.82,4.55,NaN


These three criteria share one pass over the 25,000 networks; 
`report.landscape` keeps your value and runtime for each of them for later analysis.

The sensitivity and robustness criterion is left out above because its noise
tolerance re-scores the whole landscape up to 160 times. For the full report you can run the following:

```python
report = sb.evaluate(degree_ks, n_jobs=-1)   # all five criteria, all cores
```

With `n_jobs` other than 1, please define the measure in an importable module, not in
the notebook.